# Lab 7 – NLP Document Preprocessing & TF-IDF

**Name:** Gaurav Vishal Doshi &nbsp;&nbsp; **Roll No:** 27

### Aim
- Apply Tokenization, POS Tagging, Stop-words Removal, Stemming, and Lemmatization.
- Compute Term Frequency – Inverse Document Frequency (TF-IDF) representation.

In [1]:
# Cell 1 – Importing Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import re

pd.options.display.max_columns = None
sns.set(style='whitegrid', palette='muted')

In [4]:
# Cell 2 – NLTK Downloads
import nltk, ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'wordnet_ic',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

print('NLTK data ready.')

ModuleNotFoundError: No module named 'nltk'

In [ ]:
# Cell 3 – Create Sample Documents
sample_documents = [
    "Natural language processing is a subfield of linguistics, computer science, and artificial intelligence.",
    "Machine learning is the study of computer algorithms that improve automatically through experience.",
    "Data science combines statistics, programming, and domain expertise to extract insights from data.",
    "Artificial intelligence aims to create machines that can perform tasks that typically require human intelligence.",
    "Text mining is the process of deriving high-quality information from text.",
    "Sentiment analysis determines the emotional tone behind a series of words.",
    "Named entity recognition identifies and classifies named entities in text.",
    "Topic modeling discovers abstract topics in a collection of documents.",
    "Word embeddings represent words as vectors in a continuous vector space.",
    "Chatbots use natural language processing to simulate human conversation.",
]

df = pd.DataFrame({'document': sample_documents})
df.head()

In [ ]:
# Cell 4 – EDA: Shape, Info, Describe
print('Shape:', df.shape)
print('Size :', df.size)
print()

df.info()

df['text_length'] = df['document'].apply(len)
print()
print('Text length stats:')
print(df['text_length'].describe())

In [ ]:
# Cell 5 – Sample Preview
print('Sample documents:')
for i, doc in enumerate(df['document'][:3]):
    print(f'  Doc {i+1}: {doc[:90]}...')

In [ ]:
# Cell 6 – Insert Null / Empty Rows (simulate dirty data)
null_rows = pd.DataFrame([
    {'document': 'This is a sample document with some content.'},
    {'document': ''},
    {'document': np.nan},
])

insert_pos = 3
df_with_null = pd.concat(
    [df.iloc[:insert_pos], null_rows, df.iloc[insert_pos:]]
).reset_index(drop=True)

print('New shape with inserted rows:', df_with_null.shape)
print('\nNull counts before removal:')
print(df_with_null.isnull().sum())
print('Empty documents:', (df_with_null['document'] == '').sum())
df_with_null.head(10)

In [ ]:
# Cell 7 – Remove Null / Empty Values
df_no_null = df_with_null.dropna().copy()
df_no_null = df_no_null[df_no_null['document'] != ''].reset_index(drop=True)

print('Shape after cleanup:', df_no_null.shape)
print('\nNull counts after cleanup:')
print(df_no_null.isnull().sum())
print('Empty documents after cleanup:', (df_no_null['document'] == '').sum())

In [ ]:
# Cell 8 – Visualisation: Text Length & Word Count
df_no_null['word_count']  = df_no_null['document'].apply(lambda x: len(x.split()))
df_no_null['text_length'] = df_no_null['document'].apply(len)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df_no_null['text_length'], bins=10, edgecolor='black')
axes[0].set_title('Text Length Distribution')
axes[0].set_xlabel('Length')
axes[0].set_ylabel('Frequency')

axes[1].hist(df_no_null['word_count'], bins=10, edgecolor='black')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

sns.boxplot(data=df_no_null[['text_length', 'word_count']], ax=axes[2])
axes[2].set_title('Boxplot of Text Stats')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 – Outlier Removal on Text Length (IQR)
df_clean = df_no_null.copy()

q1    = df_clean['text_length'].quantile(0.25)
q3    = df_clean['text_length'].quantile(0.75)
iqr   = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df_clean = df_clean[
    (df_clean['text_length'] >= lower) & (df_clean['text_length'] <= upper)
].reset_index(drop=True)

print('Shape after outlier cleanup:', df_clean.shape)

plt.figure(figsize=(8, 4))
sns.boxplot(data=df_clean[['text_length', 'word_count']])
plt.title('Boxplot after Outlier Removal')
plt.show()

In [ ]:
# Cell 10 – Add Document Type & Encode
df_clean['doc_type'] = ['technical'] * 5 + ['general'] * (len(df_clean) - 5)

print('Document types:')
print(df_clean['doc_type'].value_counts())

le = LabelEncoder()
df_clean['doc_type_num'] = le.fit_transform(df_clean['doc_type'])
print('\nType mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Cell 11 – Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df_clean['cleaned_text'] = df_clean['document'].apply(clean_text)
df_clean[['document', 'cleaned_text']].head()

In [ ]:
# Cell 12 – Task 1: Tokenization
sample_doc = df_clean['cleaned_text'].iloc[0]
print('Sample document:')
print(sample_doc)
print()

tokens = word_tokenize(sample_doc)
print('Tokenized tokens:')
print(tokens)

In [ ]:
# Cell 13 – POS Tagging
pos_tags = pos_tag(tokens)
print('POS Tags:')
print(pos_tags)

In [ ]:
# Cell 14 – Stop-words Removal
stop_words = set(stopwords.words('english'))
filtered_tokens = [w for w in tokens if w not in stop_words]

print('After stop-words removal:')
print(filtered_tokens)

In [ ]:
# Cell 15 – Stemming
stemmer = PorterStemmer()
stemmed = [stemmer.stem(w) for w in filtered_tokens]

print('After Stemming:')
print(stemmed)

In [ ]:
# Cell 16 – Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w) for w in filtered_tokens]

print('After Lemmatization:')
print(lemmatized)

In [ ]:
# Cell 17 – Task 2: TF-IDF Matrix
documents = df_clean['cleaned_text'].tolist()

tfidf_vec    = TfidfVectorizer(stop_words='english', max_features=100)
tfidf_matrix = tfidf_vec.fit_transform(documents)
feature_names = tfidf_vec.get_feature_names_out()

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

print('TF-IDF Matrix shape:', tfidf_matrix.shape)
print('Top 10 terms:', feature_names[:10])
tfidf_df

In [ ]:
# Cell 18 – TF-IDF Heatmap (Top 20 Terms)
top_terms = feature_names[:20]

plt.figure(figsize=(12, 6))
sns.heatmap(tfidf_df[top_terms], annot=False, cmap='YlOrRd', cbar=True)
plt.title('TF-IDF Heatmap – Top 20 Terms')
plt.xlabel('Terms')
plt.ylabel('Documents')
plt.show()

In [ ]:
# Cell 19 – Term Frequency Bar Chart
term_freq = tfidf_df.sum(axis=0).sort_values(ascending=False)

plt.figure(figsize=(12, 5))
term_freq.head(20).plot(kind='bar')
plt.title('Top 20 Terms by TF-IDF Sum')
plt.xlabel('Terms')
plt.ylabel('TF-IDF Sum')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()